# Generate output and evaluate models on held-out test set

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH, SEED
from src.utils import (
    BEST_PARAM_DICT,
    export_data,
)
import pandas as pd
from src.eval_test import get_full_reduced_scores, aggregate_result_lists
from src.sig import prep_train_test_model
from MLstatkit import Bootstrapping

In [ ]:
train_data_dir = BASE_PATH / "data"
test_data_dir = BASE_PATH / "data/test_data/cleaned"
model_dir = BASE_PATH / "models"
result_export_path = BASE_PATH / "results/test"
pred_dir = result_export_path / "predictions"

SHARED_DICT = {
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}
N_BOOTS = 1000
remove_existing = True
add_new_st = False

In [ ]:
single_scores = []
paired_scores = []
for residual_setup, sub_dict in BEST_PARAM_DICT.items():
    for impact_type, param_dict in sub_dict.items():
        if add_new_st is False and impact_type == "ST-new":
            continue
        impact_type_clean = impact_type.replace("-new", "")
        ## Create and train model
        X_train, y_train, X_test, y_test, model = prep_train_test_model(
            selected_params=param_dict,
            impact_type=impact_type_clean,
            train_data_dir=train_data_dir,
            test_data_dir=test_data_dir,
            shared_dict=SHARED_DICT,
        )
        model.fit(X_train, y_train)
        export_data(
            data_to_export=model,
            export_path=model_dir / f"{residual_setup}_{impact_type}.joblib",
            remove_existing=remove_existing,
        )
        ## GET PREDICTIONS/SCORES
        # Train
        y_pred_train_df = model.predict_all(X_train)
        export_data(
            data_to_export=y_pred_train_df,
            export_path=pred_dir / f"{residual_setup}_{impact_type}_train.tsv",
            remove_existing=remove_existing,
        )
        train_score_dict = get_full_reduced_scores(
            residual_setup=residual_setup,
            pred_df=y_pred_train_df,
            y_true=y_train,
            n_bootstraps=N_BOOTS,
            show_progress=False,
            seed=SEED,
        )

        # Test
        y_pred_test_df = model.predict_all(X_test)
        export_data(
            data_to_export=y_pred_test_df,
            export_path=pred_dir / f"{residual_setup}_{impact_type}_test.tsv",
            remove_existing=remove_existing,
        )
        test_score_dict = get_full_reduced_scores(
            residual_setup=residual_setup,
            pred_df=y_pred_test_df,
            y_true=y_test,
            n_bootstraps=N_BOOTS,
            show_progress=False,
            seed=SEED,
        )

        single_scores, paired_scores = aggregate_result_lists(
            single_scores,
            paired_scores,
            train_score_dict,
            test_score_dict,
            residual_setup,
            impact_type,
        )

single_result_df = pd.DataFrame(single_scores)
paired_result_df = pd.DataFrame(paired_scores)

In [ ]:
paired_result_df

In [ ]:
single_result_df

## Hypothesis 2

- ST - SQ

In [ ]:
for impact_type in ["self", "parent"]:
    impact_type_new = impact_type
    _, y_train, _, y_test, _ = prep_train_test_model(
        # only selecting y, so specific set up doesnt matter here
        # impact_type DOES matter though
        selected_params=BEST_PARAM_DICT["SQT"][impact_type],
        impact_type=impact_type,
        train_data_dir=train_data_dir,
        test_data_dir=test_data_dir,
        shared_dict=SHARED_DICT,
    )
    pred_train_SQT = pd.read_csv(
        pred_dir / f"SQT_{impact_type}_train.tsv", sep="\t", index_col=0
    )
    pred_train_ST = pd.read_csv(
        pred_dir / f"ST_{impact_type_new}_train.tsv", sep="\t", index_col=0
    )

    pred_test_SQT = pd.read_csv(
        pred_dir / f"SQT_{impact_type}_test.tsv", sep="\t", index_col=0
    )
    pred_test_ST = pd.read_csv(
        pred_dir / f"ST_{impact_type_new}_test.tsv", sep="\t", index_col=0
    )

    assert pred_train_SQT["DAWBA_ID"].equals(pred_train_ST["DAWBA_ID"])
    assert pred_test_SQT["DAWBA_ID"].equals(pred_test_ST["DAWBA_ID"])

    _, _, diff_train, ci_lo_diff_train, ci_hi_diff_train = Bootstrapping(
        y_true=y_train,
        y_prob=pred_train_ST["pred_stage3_total"],
        y_prob_b=pred_train_SQT["pred_stage2_total"],
        metric_str="r2",
        n_bootstraps=N_BOOTS,
        confidence_level=0.95,
        show_progress=False,
        random_state=SEED,
    )

    _, _, diff_test, ci_lo_diff_test, ci_hi_diff_test = Bootstrapping(
        y_true=y_test,
        y_prob=pred_test_ST["pred_stage3_total"],  # ST
        y_prob_b=pred_test_SQT["pred_stage2_total"],  # SQ
        metric_str="r2",
        n_bootstraps=N_BOOTS,
        confidence_level=0.95,
        show_progress=False,
        random_state=SEED,
    )
    paired_result_df.loc[len(paired_result_df)] = {
        "full/reduced": "ST-SQ",
        "impact_type": impact_type_new,
        "r2_train": f"{diff_train:.4f} ({ci_lo_diff_train:.4f}, {ci_hi_diff_train:.4f})",
        "r2_test": f"{diff_test:.4f} ({ci_lo_diff_test:.4f}, {ci_hi_diff_test:.4f})",
    }

In [ ]:
paired_result_df

## Export

In [ ]:
self_mask = single_result_df["impact_type"] == "self"
single_res_self = single_result_df[self_mask].reset_index(drop=True)
single_res_parent = single_result_df[~self_mask].reset_index(drop=True)

self_mask_paired = paired_result_df["impact_type"] == "self"
paired_res_self = paired_result_df[self_mask_paired].reset_index(drop=True)
paired_res_parent = paired_result_df[~self_mask_paired].reset_index(drop=True)

In [ ]:
if add_new_st:
    export_data(
        data_to_export=paired_res_parent,
        export_path=result_export_path / "tables/scores/paired/parent_new.tsv",
        remove_existing=True,
    )
    export_data(
        data_to_export=single_res_parent,
        export_path=result_export_path / "tables/scores/single/parent_new.tsv",
        remove_existing=True,
    )
else:
    export_data(
        data_to_export=paired_res_parent,
        export_path=result_export_path / "tables/scores/paired/parent.tsv",
        remove_existing=True,
    )
    export_data(
        data_to_export=single_res_parent,
        export_path=result_export_path / "tables/scores/single/parent.tsv",
        remove_existing=True,
    )

export_data(
    data_to_export=single_res_self,
    export_path=result_export_path / "tables/scores/single/self.tsv",
    remove_existing=True,
)


export_data(
    data_to_export=paired_res_self,
    export_path=result_export_path / "tables/scores/paired/self.tsv",
    remove_existing=True,
)